<a href="https://colab.research.google.com/github/HST0077/HYOTC/blob/main/HJM_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1일 3.72%, 3개월 3.9%, 6개월 3%, 1년 3.1%, 2년 3.3%, 5년 3.45%, 10년 3.55%으로 zero curve가 형성되어 있다고 하자. HJM 모형에 따라 초기  f ̂(t_0,t_j ),j=1,…,3650 값들을 생성하고, 이를 바탕으로 f ̂(t_1,t_j ),j=1,…,3650를 발생시켜 보아라. 이 때, 모든 선도이자율의 변동성은 10%로 동일하다고 하자.

In [2]:
import numpy as np
from scipy.interpolate import CubicSpline

# =========================================================================
# [환경 세팅] 격자(Grid) 및 시간 간격 설정
# =========================================================================
# 문제 조건: j = 1, ..., 3650 개의 만기 격자점 (10년 치 일일 데이터)
M = 3650

# 1개 격자 간격(h_l)을 1일 = 1/365년으로 설정
h_value = 1.0 / 365.0
h = np.full(M + 1, h_value)  # f_{j+1} 참조 오버플로우 방지를 위해 M+1 크기로 설정

# 각 격자 j의 만기 시점 T_j 계산 (단위: 년)
# T_1 = 1/365, T_2 = 2/365, ..., T_3650 = 3650/365 (= 10.0년)
T_j = np.cumsum(h[:-1])

# =========================================================================
# [1단계] 초기 제로 곡선 보간 및 초기 선도금리 f(t_0, t_j) 생성
# =========================================================================
# 시장의 제로 금리 데이터 (만기 단위: 년)
market_tenors = np.array([1/365, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0])
market_zero_rates = np.array([0.0372, 0.0390, 0.0300, 0.0310, 0.0330, 0.0345, 0.0355])

# Natural Cubic Spline을 이용해 제로 금리 곡선을 10년까지 매끄럽게 보간
zero_spline = CubicSpline(market_tenors, market_zero_rates, bc_type='natural')
# 기울기 계산을 위한 제로 금리 함수의 1차 도함수(dR/dT) 추출
zero_derivative = zero_spline.derivative(nu=1)

# 3650개 격자점에서의 제로 금리 R과 미분값 dR/dT 계산
R_t0 = zero_spline(T_j)
dR_dT = zero_derivative(T_j)

# f(0,T) = R(0,T) + T * dR/dT 관계식으로 초기 순간선도이자율 생성
f_t0 = R_t0 + T_j * dR_dT
print(f"1. 초기 선도이자율 곡선 f(t_0, t_j) 생성 완료 (크기: {len(f_t0)})")

1. 초기 선도이자율 곡선 f(t_0, t_j) 생성 완료 (크기: 3650)


In [3]:
# =========================================================================
# [2단계] HJM Drift 파라미터 계산 (알고리즘 반영)
# =========================================================================
# 문제 조건: 모든 선도이자율의 변동성은 10% (0.10)로 동일 (단인자 모형, d=1)
volatility = 0.10

# t_1 시점(i=1) 업데이트를 위한 이산형 Drift m_j 배열 생성 (최대 남은 만기 M-1 = 3649)
m = np.zeros(M)
B_prev = 0.0
A_cum = 0.0

# 교재 Fig. 3.16의 누적 면적 합산 알고리즘 정밀 구현
for j_idx in range(M):
    h_ij = h[1 + j_idx]  # 현재 시점에서 만기 구조를 매핑하는 시간 간격 h_{i+j}

    # 변동성 누적: A_next = A_prev + sigma * h_{i+j}
    A_cum += volatility * h_ij
    B_next = A_cum * A_cum  # 자승(제곱) 합산

    # 무위험 차익거래 불가능 이산형 Drift 파라미터 산출
    m[j_idx] = (B_next - B_prev) / (2.0 * h_ij)
    B_prev = B_next

In [1]:
import numpy as np
from scipy.interpolate import CubicSpline

def solve_hjm_problem_14_extended():
    # =========================================================================
    # [환경 세팅] 격자(Grid) 및 시간 간격 설정
    # =========================================================================
    # 문제 조건: j = 1, ..., 3650 개의 만기 격자점 (10년 치 일일 데이터)
    M = 3650

    # 1개 격자 간격(h_l)을 1일 = 1/365년으로 설정
    h_value = 1.0 / 365.0
    h = np.full(M + 1, h_value)  # f_{j+1} 참조 오버플로우 방지를 위해 M+1 크기로 설정

    # 각 격자 j의 만기 시점 T_j 계산 (단위: 년)
    # T_1 = 1/365, T_2 = 2/365, ..., T_3650 = 3650/365 (= 10.0년)
    T_j = np.cumsum(h[:-1])

    # =========================================================================
    # [1단계] 초기 제로 곡선 보간 및 초기 선도금리 f(t_0, t_j) 생성
    # =========================================================================
    # 시장의 제로 금리 데이터 (만기 단위: 년)
    market_tenors = np.array([1/365, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0])
    market_zero_rates = np.array([0.0372, 0.0390, 0.0300, 0.0310, 0.0330, 0.0345, 0.0355])

    # Natural Cubic Spline을 이용해 제로 금리 곡선을 10년까지 매끄럽게 보간
    zero_spline = CubicSpline(market_tenors, market_zero_rates, bc_type='natural')
    # 기울기 계산을 위한 제로 금리 함수의 1차 도함수(dR/dT) 추출
    zero_derivative = zero_spline.derivative(nu=1)

    # 3650개 격자점에서의 제로 금리 R과 미분값 dR/dT 계산
    R_t0 = zero_spline(T_j)
    dR_dT = zero_derivative(T_j)

    # f(0,T) = R(0,T) + T * dR/dT 관계식으로 초기 순간선도이자율 생성
    f_t0 = R_t0 + T_j * dR_dT
    print(f"1. 초기 선도이자율 곡선 f(t_0, t_j) 생성 완료 (크기: {len(f_t0)})")

    # =========================================================================
    # [2단계] HJM Drift 파라미터 계산 (Fig. 3.16 알고리즘 반영)
    # =========================================================================
    # 문제 조건: 모든 선도이자율의 변동성은 10% (0.10)로 동일 (단인자 모형, d=1)
    volatility = 0.10

    # t_1 시점(i=1) 업데이트를 위한 이산형 Drift m_j 배열 생성 (최대 남은 만기 M-1 = 3649)
    m = np.zeros(M)
    B_prev = 0.0
    A_cum = 0.0

    # 교재 Fig. 3.16의 누적 면적 합산 알고리즘 정밀 구현
    for j_idx in range(M):
        h_ij = h[1 + j_idx]  # 현재 시점에서 만기 구조를 매핑하는 시간 간격 h_{i+j}

        # 변동성 누적: A_next = A_prev + sigma * h_{i+j}
        A_cum += volatility * h_ij
        B_next = A_cum * A_cum  # 자승(제곱) 합산

        # 무위험 차익거래 불가능 이산형 Drift 파라미터 산출
        m[j_idx] = (B_next - B_prev) / (2.0 * h_ij)
        B_prev = B_next

    # =========================================================================
    # [3단계] 몬테카를로 난수 발생 및 f(t_1, t_j) 시뮬레이션 업데이트 (Fig. 3.17 반영)
    # =========================================================================
    # 첫 번째 스텝(t_1)에서 발생하는 단일 요인 시장 충격 난수 Z_1 추출
    Z_1 = np.random.normal(0.0, 1.0)
    print(f"2. 발생된 1일 차 무작위 충격 난수 Z_1: {Z_1:.4f}")

    # t_1 시점(1일 후)에서 남은 만기 곡선의 최대 크기는 M - 1 = 3649개가 됨
    f_t1 = np.zeros(M - 1)
    h_i = h[0]  # 현재 흘러간 시간 단기 스텝 dt = h_1 (1/365년)

    # 교재 Fig. 3.17 알고리즘의 핵심 업데이트 루프 전개
    for j_idx in range(M - 1):
        # 단인자 변동성 충격량: S = sigma * Z_1
        S = volatility * Z_1

        # 오일러-마루야마 이산화 전진 업데이트 및 롤다운 처리
        # f_j <- f_{j+1} + m_j * h_i + S * sqrt(h_i)
        f_t1[j_idx] = f_t0[j_idx + 1] + m[j_idx] * h_i + S * np.sqrt(h_i)

    print(f"3. 1일 후의 선도이자율 곡선 f(t_1, t_j) 발생 완료 (크기: {len(f_t1)})")
    print("-" * 85)

    # =========================================================================
    # [결과 모니터링] 주요 만기 마일스톤별 금리 데이터 출력
    # =========================================================================
    print(f"{'만기 마일스톤':^12} | {'인덱스 (j)':^10} | {'초기 선도금리 f(t_0)':^20} | {'시뮬레이션 f(t_1)':^20}")
    print("-" * 85)

    # 주요 만기 시점 매핑 (1일, 3개월, 6개월, 1년, 5년, 10년 직전)
    milestone_indices = [0, 90, 181, 364, 1824, 3648]
    milestone_labels = ["1일", "3개월", "6개월", "1년", "5년", "10년(인접)"]

    for label, idx in zip(milestone_labels, milestone_indices):
        f_t1_str = f"{f_t1[idx]*100:16.4f} %" if idx < len(f_t1) else f"{'만기 소멸 (10년 경과)':^22}"
        print(f"{label:^12} | j = {idx+1:<6} | {f_t0[idx]*100:16.4f} % | {f_t1_str}")
    print("-" * 85)

if __name__ == "__main__":
    solve_hjm_problem_14_extended()

1. 초기 선도이자율 곡선 f(t_0, t_j) 생성 완료 (크기: 3650)
2. 발생된 1일 차 무작위 충격 난수 Z_1: -0.2923
3. 1일 후의 선도이자율 곡선 f(t_1, t_j) 발생 완료 (크기: 3649)
-------------------------------------------------------------------------------------
  만기 마일스톤    |  인덱스 (j)   |    초기 선도금리 f(t_0)    |     시뮬레이션 f(t_1)    
-------------------------------------------------------------------------------------
     1일      | j = 1      |           3.7256 % |           3.5837 %
    3개월      | j = 91     |           3.4395 % |           3.2552 %
    6개월      | j = 182    |           1.4407 % |           1.3017 %
     1년      | j = 365    |           4.4846 % |           4.3317 %
     5년      | j = 1825   |           4.2491 % |           4.1095 %
  10년(인접)    | j = 3649   |           3.0512 % |           2.9253 %
-------------------------------------------------------------------------------------


In [1]:
import numpy as np
from scipy.interpolate import CubicSpline

def generate_hjm_initial_forward_curve(M, h):
    """
    주어진 Zero Curve 마일스톤 데이터를 바탕으로
    HJM 시뮬레이션에 필요한 초기 순간선도이자율 f(t_0, t_j) 배열을 생성합니다.

    Parameters:
    M (int): 총 만기 격자의 개수
    h (numpy.ndarray): 각 격자 스텝 간의 시간 간격 배열 (단위: 년)

    Returns:
    numpy.ndarray: 크기가 M인 f(t_0, t_j) 선도이자율 곡선 배열
    """
    # 1. 주어진 시장의 제로 금리 데이터 세팅 (만기 기준 일치화, 단위: 년)
    # 1일=1/365, 3개월=0.25, 6개월=0.5, 1년=1.0, 2년=2.0, 5년=5.0, 10년=10.0
    market_tenors = np.array([1/365, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0])
    market_zero_rates = np.array([0.0372, 0.0390, 0.0300, 0.0310, 0.0330, 0.0345, 0.0355])

    # 2. 제로 금리를 매끄럽게 연결하기 위해 Cubic Spline 보간기 생성
    # bc_type='natural'을 주어 양 끝단 외삽이 튕기지 않고 안정되도록 설정
    zero_curve_spline = CubicSpline(market_tenors, market_zero_rates, bc_type='natural')

    # 제로 곡선의 미분 수식(기출기 1차 도함수) 자동 추출
    zero_curve_derivative = zero_curve_spline.derivative(nu=1)

    # 3. HJM 격자에 맞는 만기 타임라인 T_j 구성 (누적 시간 계산)
    # j = 1, ..., M에 매핑되는 만기 시점들
    T_j = np.cumsum(h)

    # 시뮬레이션 격자가 보간 범위를 벗어나는 것을 방지 (최대 10년 제한 마스킹)
    T_j = np.clip(T_j, 1/365, 10.0)

    # 4. HJM 공식 적용: f(t_0, T) = R(t_0, T) + T * dR/dT
    R = zero_curve_spline(T_j)          # 격자점에서의 제로 금리
    dR_dT = zero_curve_derivative(T_j)  # 격자점에서의 제로 금리 미분값

    f_t0_tj = R + T_j * dR_dT

    return T_j, f_t0_tj

# ==========================================
# 실행 및 초기 곡선 데이터 시각화 확인
# ==========================================
if __name__ == "__main__":
    # 시뮬레이션 뼈대 세팅 (Fig. 3.17 연계용)
    M = 11  # t_0부터 t_10까지 총 11개 만기 지점
    h = np.array([0.5] * M)  # 6개월(0.5년) 간격의 만기 구조 격자

    # HJM 초기 선도금리 곡선 계산 구동
    tenors, initial_forward_curve = generate_hjm_initial_forward_curve(M, h)

    # 결과 정밀 출력
    print("📊 [시장 Zero Curve 반영] HJM 초기 순간선도이자율 f(t_0, t_j) 생성 결과")
    print("-" * 65)
    print(f"{'만기 인덱스 (j)':^15} | {'만기 시점 (년)':^15} | {'선도이자율 f(t_0, t_j)':^25}")
    print("-" * 65)
    for idx in range(M):
        print(f"j = {idx+1:<10} | {tenors[idx]:^15.2f} | {initial_forward_curve[idx]*100:18.4f} %")
    print("-" * 65)
    print("\nHJM 시뮬레이터(f_initial)에 주입할 최종 NumPy 배열:\n", initial_forward_curve)

📊 [시장 Zero Curve 반영] HJM 초기 순간선도이자율 f(t_0, t_j) 생성 결과
-----------------------------------------------------------------
  만기 인덱스 (j)    |    만기 시점 (년)    |     선도이자율 f(t_0, t_j)    
-----------------------------------------------------------------
j = 1          |      0.50       |             1.4469 %
j = 2          |      1.00       |             4.4846 %
j = 3          |      1.50       |             3.4516 %
j = 4          |      2.00       |             2.7091 %
j = 5          |      2.50       |             2.8939 %
j = 6          |      3.00       |             3.2122 %
j = 7          |      3.50       |             3.5855 %
j = 8          |      4.00       |             3.9350 %
j = 9          |      4.50       |             4.1824 %
j = 10         |      5.00       |             4.2491 %
j = 11         |      5.50       |             4.1796 %
-----------------------------------------------------------------

HJM 시뮬레이터(f_initial)에 주입할 최종 NumPy 배열:
 [0.01446932 0.04484622 0.0345

In [2]:
import numpy as np
from hjm_bonds import (
  HJMConfig, constant_sigma, initialize_forward_curve,
  zero_coupon_cashflows, coupon_bond_cashflows, price_bond_mc,
)

maturities = np.linspace(0, 5, 6)          # t_0,...,t_5
f0 = initialize_forward_curve(maturities, lambda t: 0.05)

config = HJMConfig(
    maturities=maturities,
    sigma=constant_sigma(0.01),  # Example 3.6.1
    seed=42,
)

# 제로쿠폰 (만기 t_3)
cf_zcb = zero_coupon_cashflows(maturity_idx=3, n_times=5)
price, se = price_bond_mc(config, f0, cf_zcb, n_paths=50_000)

# 쿠폰채 (액면 100, 쿠폰 5, 매년 지급)
cf_coupon = coupon_bond_cashflows(
    coupon_dates=[1, 2, 3, 4, 5],
    maturity_idx=5,
    face=100.0,
    coupon=5.0,
    n_times=5,
)
price, se = price_bond_mc(config, f0, cf_coupon, n_paths=50_000)

ModuleNotFoundError: No module named 'hjm_bonds'

In [1]:
import numpy as np

def calculate_discrete_drift(s, h, i, M, d):
    """
    Fig. 3.16 알고리즘: 이산화된 Drift 파라미터 m_j 계산
    """
    m = np.zeros(M - i)
    A_prev = np.zeros(d)
    B_prev = 0.0

    # j는 남은 만기 인덱스 (1부터 M-i까지)
    for j_idx in range(M - i):
        j = j_idx + 1
        B_next = 0.0
        h_ij = h[i + j - 1] # h_{i+j}

        for k in range(d):
            # s_j(k) * h_{i+j} 누적
            A_next_k = A_prev[k] + s[j_idx, k] * h_ij
            B_next += A_next_k * A_next_k
            A_prev[k] = A_next_k

        m[j_idx] = (B_next - B_prev) / (2.0 * h_ij)
        B_prev = B_next

    return m

def hjm_bond_simulation(f_initial, h, coupon_dates, coupon_rate, face_value, d, num_paths):
    """
    Fig. 3.17 알고리즘 기반: HJM 선도금리 변동 시뮬레이션 및 채권 가격 평가
    """
    M = len(f_initial)
    total_C = 0.0

    # 몬테카를로 시뮬레이션 경로 루프
    for path in range(num_paths):
        # 각 경로마다 선도금리 곡선 초기화
        f = f_initial.copy()
        D = 1.0  # 누적 할인 팩터
        C = 0.0  # 누적 현재가치

        # 시간 축 루프 (i = 1, ..., M-1)
        for i in range(1, M):
            h_i = h[i - 1]

            # 1. 단기금리(f_1)를 이용한 할인 팩터 업데이트
            D *= np.exp(-f[0] * h_i)

            # 2. 변동성 s_j(k) 평가 (여기서는 예시로 고정된 결정론적 변동성 구조 사용)
            # 남은 만기 개수(M-i)만큼의 변동성 행렬 생성
            current_M_minus_i = M - i
            s = np.zeros((current_M_minus_i, d))
            for j_idx in range(current_M_minus_i):
                for k in range(d):
                    # 만기가 길어질수록 변동성이 감쇄하는 예시 모델 (상수 처리 가능)
                    s[j_idx, k] = 0.01 * np.exp(-0.1 * j_idx) / (k + 1)

            # 3. Fig. 3.16을 이용해 drift m_j 계산
            m = calculate_discrete_drift(s, h, i, M, d)

            # 4. d개의 독립 표준정규분포 난수 생성
            Z = np.random.normal(0.0, 1.0, d)

            # 5. 남은 만기 축(j = 1, ..., M-i) 루프 돌며 선도금리 업데이트
            f_next = np.zeros(current_M_minus_i)
            for j_idx in range(current_M_minus_i):
                S = 0.0
                for k in range(d):
                    S += s[j_idx, k] * Z[k]

                # f_j <- f_{j+1} + m_j * h_i + S * sqrt(h_i)
                f_next[j_idx] = f[j_idx + 1] + m[j_idx] * h_i + S * np.sqrt(h_i)

            # 금리 곡선 갱신 (크기가 매 스텝 1씩 줄어듦)
            f = f_next

            # 6. Example 3.6.4: 현금흐름 P 설정
            current_date_idx = i  # 현재 시점 t_i
            P = 0.0

            # 이표 지급일 검증
            if current_date_idx in coupon_dates:
                P += coupon_rate
            # 만기일 검증 (최종 시점)
            if current_date_idx == M - 1:
                P += face_value

            # 7. 현재가치 누적
            C += D * P

        total_C += C

    # 모든 경로의 평균값 반환
    return total_C / num_paths

# ==========================================
# 테스트 실행 예제
# ==========================================
if __name__ == "__main__":
    # 시뮬레이션 환경 설정
    M = 11  # 총 만기 수 (t_0부터 t_10까지)
    h = np.array([0.5] * M)  # 모든 시간 간격 h_l = 0.5년 (6개월 스텝)

    # 1. 초기 선도금리 곡선 (플랫하게 4%로 가정)
    f_initial = np.array([0.04] * M)

    # 2. 채권 조건 (Example 3.6.4 반영)
    # 5년 만기 이표채 (h=0.5이므로 인덱스 10이 5년 만기 시점)
    # 매 1년마다(2스텝마다) 쿠폰 지급한다고 가정: 인덱스 2, 4, 6, 8, 10
    coupon_dates = [2, 4, 6, 8, 10]
    coupon_rate = 5.0      # 이표금액 5
    face_value = 100.0     # 액면가 100

    d = 2            # 2인자 HJM 모형
    num_paths = 5000 # 몬테카를로 경로 수

    # HJM 시뮬레이션을 통한 채권 가격 계산
    simulated_bond_price = hjm_bond_simulation(
        f_initial, h, coupon_dates, coupon_rate, face_value, d, num_paths
    )

    # 3. 검증을 위한 오늘 자 기준 이론적 채권 가격 (초기 선도금리로 바로 할인)
    theoretical_price = 0.0
    discount_factor = 1.0
    for i in range(1, M):
        discount_factor *= np.exp(-f_initial[0] * h[i-1])
        P = 0.0
        if i in coupon_dates:
            P += coupon_rate
        if i == M - 1:
            P += face_value
        theoretical_price += discount_factor * P

    print(f"📊 [HJM 시뮬레이션] 채권 가격 (Paths={num_paths}): {simulated_bond_price:.4f}")
    print(f"📈 [초기 곡선 기준] 이론적 채권 가격              : {theoretical_price:.4f}")
    print(f"📝 오차(Error)                                 : {abs(simulated_bond_price - theoretical_price):.4f}")

📊 [HJM 시뮬레이션] 채권 가격 (Paths=5000): 104.0620
📈 [초기 곡선 기준] 이론적 채권 가격              : 104.0816
📝 오차(Error)                                 : 0.0196
